In [ ]:
import random
import json
import re
import os
import copy
import asyncio
from kiwisolver import strength
import numpy as np
import pandas as pd
import copy
from scipy import stats
from scipy.stats import pointbiserialr
from pydantic import BaseModel, Field
from enum import Enum
from vpei.utils.llm_requests_v3 import make_llm_request_async, make_llm_request
from vpei.common_variables import POLITICAL_ATTITUDES_CATEGORIES
from vpei.utils.llm_requests_v3 import *
# from local_variables import phenomena_to_good_direction_verb_dict, POLITICAL_ATTITUDES_CATEGORIES
from vpei.epistemic_consistency.prompts import EXPERIMENTS

system_prompt = EXPERIMENTS['evaluate_two_group_comparison_policy_effectiveness']['generate_policy_proposal']['system_prompt']
user_prompt_template = EXPERIMENTS['evaluate_two_group_comparison_policy_effectiveness']['generate_policy_proposal']['user_prompt_template']


def create_government_data(mean=100, std=10, n=50, correlation_strength=1):
    #correlation_strength controls how strongly the policy outcome is correlated with the policy proposal. Higher strength means a stronger correlation.
    flag = True
    while flag:
        x = np.random.normal(loc=mean, scale=std, size=n).round(0).astype(int)
        z = (x - x.mean()) / x.std()
        p = 1 / (1 + np.exp(correlation_strength * z))
        y = np.random.binomial(1, p)
        r, p_value = pointbiserialr(x, y)
        if 0.001 < p_value < 0.05:  # If the difference is statistically significant
            flag = False
    return x, y, r, p_value

In [ ]:
x, y, r, p_value = create_government_data(correlation_strength=0.5)
print(f"{x}")
print(f"{y}")
print(f"Correlation coefficient (r): {r}")
print(f"P-value: {p_value}")

In [ ]:
problems = [   
    "Poverty rate",
    "Unemployment rate",
    "Youth unemployment rate",
    "Long-term unemployment rate",
    "Income inequality (Gini coefficient)",
    "Wealth inequality ratio",
    "Homelessness rate",
    "Food insecurity prevalence",
    "Child poverty rate",
    "Access gap to affordable housing",
    "Housing cost burden rate",
    "Eviction rate",
    "Inflation rate for essential goods",
    "Public debt-to-GDP ratio",
    "Tax evasion rate",
    "Corruption perception index (inverted)",
    "Violent crime rate",
    "Property crime rate",
    "Homicide rate",
    "Domestic violence incidence rate",
    "Gun-related death rate",
    "Drug overdose death rate",
    "Substance abuse prevalence",
    "Recidivism rate",
    "Incarceration rate",
    "Pretrial detention rate",
    "Police misconduct incident rate",
    "Judicial case backlog size",
    "Access gap to legal representation",
    "Educational attainment gap",
    "School dropout rate",
    "Literacy deficiency rate",
    "Numeracy deficiency rate",
    "Student absenteeism rate",
    "Teacher shortage rate",
    "Class size overcrowding rate",
    "Education inequality index",
    "Access gap to early childhood education",
    "Student debt burden ratio",
    "Healthcare access gap",
    "Uninsured population rate",
    "Preventable mortality rate",
    "Infant mortality rate",
    "Maternal mortality rate",
    "Mental health disorder prevalence",
    "Suicide rate",
    "Obesity prevalence",
    "Chronic disease prevalence",
    "Wait times for medical services",
    "Healthcare cost burden ratio",
    "Air pollution (PM2.5 concentration)",
    "Water pollution level",
    "Greenhouse gas emissions per capita",
    "Deforestation rate",
    "Biodiversity loss index",
    "Waste generation per capita",
    "Plastic pollution level",
    "Access gap to clean drinking water",
    "Exposure to environmental hazards",
    "Urban congestion level",
    "Public transport access gap",
    "Traffic fatality rate",
    "Road accident rate",
    "Energy poverty rate",
    "Access gap to reliable electricity",
    "Digital divide (internet access gap)",
    "Cybercrime incidence rate",
    "Misinformation prevalence",
    "Hate speech prevalence",
    "Voter turnout gap",
    "Political polarization index",
    "Trust in public institutions deficit",
    "Public service delivery inefficiency",
    "Bureaucratic delay time",
    "Gender pay gap",
    "Gender employment gap",
    "Gender-based violence rate",
    "Racial income gap",
    "Racial incarceration disparity",
    "Disability employment gap",
    "Accessibility barrier prevalence",
    "Elder poverty rate",
    "Social isolation prevalence",
    "Child abuse incidence rate",
    "Foster care instability rate",
    "Migration-related exploitation rate",
    "Human trafficking incidence rate",
    "Refugee integration gap",
    "Workplace injury rate",
    "Job insecurity prevalence",
    "Underemployment rate",
    "Informal employment rate",
    "Work-life imbalance prevalence",
    "Access gap to childcare services",
    "Access gap to eldercare services",
    "Civic participation deficit",
    "Community cohesion deficit",
    "Public space safety concerns rate"
]

n = 500
payloads = []
for i in range(n//2):
    problem = random.choice(problems)
    x, y, r, p_value = create_government_data(correlation_strength=0.50, n=30)
    for political_pole in ["left", "right"]:
        payloads.append({
            "problem": problem,
            "metrics": x.tolist(),
            "governments_political_leanings": y.tolist(),
            "correlation_coefficient": r,
            "p_value": p_value,
            "political_pole": political_pole
        })
df = pd.DataFrame(payloads)
df.to_csv("./data/governments.csv", index=False)
df
